In [11]:
import numpy as np
import torch
import pyro
import pyro.distributions as dist
from scipy.integrate import dblquad
from scipy.stats import multivariate_normal

In [9]:
def pyro_model_(data, parameter_len, epsilon, prior_sigma):
    prior_parameter = pyro.sample("prior_parameter", dist.MultivariateNormal(torch.zeros(parameter_len), torch.eye(parameter_len)*prior_sigma))
    mean = torch.tensor([prior_parameter[0] - torch.max(prior_parameter), prior_parameter[1]]).float()
    with pyro.plate("data_plate"):
        pyro.sample("obs", dist.MultivariateNormal(mean, (epsilon**2)*torch.eye(len(data))), obs=data)    


def run_HMC(data, parameter_len, epsilon, prior_sigma, stepsize, num_samples, initial_params, warmup_steps):

    pyro.clear_param_store()
    pyro_model = lambda data: pyro_model_(data=data, parameter_len=parameter_len, epsilon=epsilon, prior_sigma=prior_sigma)
    pyro_kernel =  pyro.infer.mcmc.HMC(model=pyro_model, step_size=stepsize)
    pyro_mcmc = pyro.infer.mcmc.MCMC(kernel=pyro_kernel, num_samples=num_samples, initial_params={'prior_parameter': initial_params}, warmup_steps=warmup_steps)
    pyro_mcmc.run(data)
    samples = pyro_mcmc.get_samples()["prior_parameter"]
                    
    return samples

data = torch.tensor([-1.,-1.])
parameter_len = 2
prior_sigma = 1.
stepsize = 1.
warmup_steps = 10

eps = 0.1
num_samples = 100
initial_params = torch.tensor([0.,0.])




In [43]:
def ABC_prob(epsilon, prior_sigma, data):
    def fn_(y, x, epsilon, prior_sigma):
        #y is theta2, x is theta1
        lh = multivariate_normal.pdf(data, mean=np.array([x - np.maximum(x,y), y]), cov=epsilon**2)
        pr = multivariate_normal.pdf(np.array([x,y]), mean=np.array([0.,0.]), cov=prior_sigma**2)
        return lh*pr
    fn = lambda y, x: fn_(y, x, epsilon=epsilon, prior_sigma=prior_sigma)
    numerator = dblquad(fn, -np.inf, np.inf, lambda x:x, np.inf)[0]
    denominator = dblquad(fn, -np.inf, np.inf, -np.inf, np.inf)[0]
    return numerator / denominator

In [45]:
ABC_prob(epsilon=10, prior_sigma=1., data=data.numpy())

0.49753709550223263

In [ ]:
def ABC_error():
    pass

def MCMC_error(weights, particles):
    pass

def Overall_error(weights, particles):
    pass


In [ ]:
run_HMC(data=data, parameter_len=parameter_len, epsilon=eps, prior_sigma=prior_sigma, stepsize=stepsize, num_samples=num_samples, initial_params=initial_params, warmup_steps=warmup_steps)
